# HAMAR SHIKSHA: Gyan GPU server (Colab)

Runs the **Gyan SLM** (a Gemma 4 model) and the **bge-m3 embedding model** on this Colab GPU and exposes them through an
authenticated HTTPS tunnel, so the HAMAR SHIKSHA app (on your PC or on Render) can use them. Run the cells top to bottom.

**Test use only, with demo data.** Student questions sent to the app travel to this Colab VM. Do not use real student data.

1. `Runtime > Change runtime type > A100 GPU`.
2. Run all cells. Cell 8 prints the lines to paste into the app's `.env`.
3. **The tunnel URL (and the token, unless you set a secret) change every time this notebook is restarted.** Update `.env` (or the Render env vars) and restart the app.
4. Colab disconnects after idle time and has a maximum session length. Keep this tab open while testing.
5. Outputs contain the token. Use `Edit > Clear all outputs` before sharing or saving a copy.

Optional: add a Colab secret named `HS_GPU_TOKEN` (key icon, left sidebar) to keep the same token between runs.

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
# 2. Settings
import os, secrets

# Gyan SLM. Ollama tags (checked 2026-10): gemma4:31b-it-q4_K_M (20 GB), gemma4:31b-it-qat (19 GB),
# gemma4:31b-it-q8_0 (34 GB, 80 GB A100 only), gemma4:26b-a4b-it-q4_K_M (18 GB, MoE, faster),
# gemma4:12b-it-q4_K_M (8 GB). On a 40 GB A100 use one of the 4-bit 31B tags.
MODEL = "gemma4:31b-it-q4_K_M"
EMBED_MODEL = "bge-m3"          # multilingual (Hindi/English) embeddings, used for hybrid retrieval

try:
    from google.colab import userdata
    TOKEN = userdata.get("HS_GPU_TOKEN")
except Exception:
    TOKEN = None
TOKEN = TOKEN or secrets.token_urlsafe(32)
print("model:", MODEL, "| embeddings:", EMBED_MODEL)

In [ ]:
# 3. Install and start Ollama (about 1-2 minutes)
!apt-get install -y -qq zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_install.log 2>&1; tail -n 3 /tmp/ollama_install.log

import subprocess, time, urllib.request
env = {**os.environ, "OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_KEEP_ALIVE": "-1",
       "OLLAMA_NUM_PARALLEL": "2", "OLLAMA_FLASH_ATTENTION": "1"}
ollama_log = open("/tmp/ollama_serve.log", "w")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=ollama_log, stderr=ollama_log)
for _ in range(60):
    try:
        print("ollama up, version:", urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2).read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start; see /tmp/ollama_serve.log")

In [ ]:
# 4. Download the models (about 20 GB for the 31B 4-bit model; a few minutes)
for m in (EMBED_MODEL, MODEL):
    print("pulling", m)
    subprocess.run(["ollama", "pull", m], env=env, check=True)
!ollama list

In [ ]:
%%writefile gpu_proxy.py
"""Authenticating front door for an Ollama server running in Colab.

Ollama has no authentication, and a tunnel makes it reachable from the internet, so the tunnel
points here instead. Every request must carry `Authorization: Bearer <GPU_TOKEN>`; only the
read/inference endpoints HAMAR SHIKSHA uses are forwarded (no pull / delete / create / push),
so a leaked token cannot change what is installed on the GPU box. Request bodies are never logged.

The notebook `hamar_gyan_gpu.ipynb` embeds this file; keep the two in step.
Run standalone:  GPU_TOKEN=... UPSTREAM=http://127.0.0.1:11434 uvicorn gpu_proxy:app --port 8080
"""
from __future__ import annotations

import hmac
import os

import httpx
from fastapi import FastAPI, Request, Response

UPSTREAM = os.environ.get("UPSTREAM", "http://127.0.0.1:11434").rstrip("/")
TOKEN = os.environ.get("GPU_TOKEN", "")
ALLOWED = {("POST", "/api/chat"), ("POST", "/api/embed"), ("POST", "/api/embeddings"),
           ("GET", "/api/tags"), ("GET", "/api/version")}
HOP = {"host", "content-length", "connection", "authorization", "accept-encoding", "transfer-encoding"}

app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
client = httpx.AsyncClient(base_url=UPSTREAM, timeout=600)


@app.get("/health")
async def health():
    return {"status": "ok"}  # unauthenticated: lets the notebook check the tunnel; reveals nothing


@app.api_route("/{path:path}", methods=["GET", "POST"])
async def forward(path: str, request: Request):
    if not TOKEN:
        return Response("proxy has no GPU_TOKEN configured", status_code=503)
    sent = request.headers.get("authorization", "")
    if not hmac.compare_digest(sent.encode(), f"Bearer {TOKEN}".encode()):
        return Response("unauthorized", status_code=401)
    if (request.method, "/" + path) not in ALLOWED:
        return Response("endpoint not allowed", status_code=403)
    headers = {k: v for k, v in request.headers.items() if k.lower() not in HOP}
    try:
        r = await client.request(request.method, "/" + path, content=await request.body(), headers=headers)
    except httpx.HTTPError:
        return Response("model server unreachable", status_code=502)
    out = {k: v for k, v in r.headers.items() if k.lower() in ("content-type",)}
    return Response(r.content, status_code=r.status_code, headers=out)

In [ ]:
# 6. Start the proxy
# Not 8080: Colab's own backend listens there, so the proxy could never bind and the health check
# would be answered by Colab instead of the proxy.
!pip install -q fastapi uvicorn httpx
import socket, sys
PROXY_PORT = 18080
if not os.path.exists("gpu_proxy.py"):
    raise RuntimeError("gpu_proxy.py is missing: run the previous cell (%%writefile gpu_proxy.py) first")
with socket.socket() as s:
    if s.connect_ex(("127.0.0.1", PROXY_PORT)) == 0:
        raise RuntimeError(f"port {PROXY_PORT} is already in use: set PROXY_PORT to another number in this cell")
proxy_log = open("/tmp/proxy.log", "w")
proxy_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "gpu_proxy:app", "--host", "127.0.0.1", "--port", str(PROXY_PORT),
     "--log-level", "warning"],
    env={**os.environ, "GPU_TOKEN": TOKEN, "UPSTREAM": "http://127.0.0.1:11434"}, stdout=proxy_log, stderr=proxy_log)
up = False
for _ in range(30):
    try:
        print("proxy up:", urllib.request.urlopen(f"http://127.0.0.1:{PROXY_PORT}/health", timeout=2).read().decode())
        up = True
        break
    except Exception:
        if proxy_proc.poll() is not None:   # the process already exited: no point waiting
            break
        time.sleep(1)
if not up:
    proxy_log.flush()
    print("---- /tmp/proxy.log ----")
    print(open("/tmp/proxy.log").read() or "(empty)")
    raise RuntimeError("proxy did not start; the log above says why")

In [ ]:
# 6b. Load the models into GPU memory BEFORE anything goes through the tunnel
# A cold load of the 31B model takes minutes, and Cloudflare gives up on any response slower than about
# 100 s (HTTP 524). So the first request through the tunnel must find the model already loaded, with the
# same context size the app will ask for (a different num_ctx makes Ollama reload the model).
import requests
CTX = 8192   # = HS_SLM_CONTEXT_TOKENS in the app
LOCAL = "http://127.0.0.1:11434"
t = time.time()
requests.post(LOCAL + "/api/embed", json={"model": EMBED_MODEL, "input": ["warm up"]}, timeout=900).raise_for_status()
print("embedding model ready in %.0f s" % (time.time() - t))

def warm_chat(extra):
    return requests.post(LOCAL + "/api/chat", timeout=1800, json={
        "model": MODEL, "stream": False, "options": {"num_predict": 8, "num_ctx": CTX},
        "messages": [{"role": "user", "content": "Say OK."}], **extra})

t = time.time()
THINK = {"think": False}   # reasoning models would spend the answer budget on hidden thinking
r = warm_chat(THINK)
if r.status_code == 400 and "think" in r.text.lower():   # this model has no thinking mode
    print("model rejects the think field; continuing without it:", r.text[:120])
    THINK = {}
    r = warm_chat(THINK)
r.raise_for_status()
print("chat model ready in %.0f s" % (time.time() - t))
!ollama ps
print("PROCESSOR should read 100% GPU. A CPU share means the model does not fit this GPU: pick a smaller MODEL.")

In [ ]:
# 7. Public HTTPS tunnel (Cloudflare quick tunnel, no account needed)
import re
!curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
cf_log = open("/tmp/cloudflared.log", "w")
cf_proc = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PROXY_PORT}"],
                           stdout=cf_log, stderr=cf_log)
PUBLIC_URL = None
for _ in range(60):
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/tmp/cloudflared.log").read())
    if m:
        PUBLIC_URL = m.group(0)
        break
    time.sleep(1)
if not PUBLIC_URL:
    raise RuntimeError("no tunnel URL; see /tmp/cloudflared.log")
for _ in range(30):  # DNS for a new quick tunnel can take a few seconds
    try:
        print("tunnel health:", urllib.request.urlopen(PUBLIC_URL + "/health", timeout=5).read().decode())
        break
    except Exception:
        time.sleep(2)
print(PUBLIC_URL)

In [ ]:
# 8. Smoke test through the public URL, then print the app settings
import requests
H = {"Authorization": f"Bearer {TOKEN}"}
assert requests.post(PUBLIC_URL + "/api/chat", json={}).status_code == 401, "proxy must reject requests without the token"

t = time.time()
r = requests.post(PUBLIC_URL + "/api/chat", headers=H, timeout=600, json={
    "model": MODEL, "stream": False, "options": {"num_predict": 120, "temperature": 0.2, "num_ctx": CTX},
    "messages": [{"role": "user", "content": "भिन्न में हर क्या होता है? दो वाक्यों में बताइए।"}], **THINK})
r.raise_for_status()
print("chat %.1fs ->" % (time.time() - t), r.json()["message"]["content"])
# If that answer is empty or shows reasoning text instead of an answer, the model is in a thinking mode: tell Claude.

t = time.time()
e = requests.post(PUBLIC_URL + "/api/embed", headers=H, timeout=120, json={"model": EMBED_MODEL, "input": ["fractions", "भिन्न"]})
e.raise_for_status()
print("embed %.1fs, dims=%d" % (time.time() - t, len(e.json()["embeddings"][0])))

print(f'''
================ paste into HAMAR_Shiksha/.env (or Render env vars) ================
HS_SLM_RUNTIME=ollama
HS_SLM_URL={PUBLIC_URL}
HS_SLM_API_KEY={TOKEN}
HS_SLM_MODEL={MODEL}
HS_EMBEDDING_MODEL=ollama:{EMBED_MODEL}
HS_SLM_CONTEXT_TOKENS=8192
HS_SLM_TIMEOUT_SECONDS=300
====================================================================================
''')

In [ ]:
# Status (run any time): what is loaded, GPU memory, process health
!ollama ps
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv
print("ollama:", ollama_proc.poll(), "proxy:", proxy_proc.poll(), "tunnel:", cf_proc.poll(), "(None = running)")